In [ ]:
# Databricks Notebook: ETL from AWS S3 to Azure Data Lake (Batch, Nested JSON)
# Author: Data Engineer
# Description: Daily batch job to process nested JSON with 60+ columns from AWS S3 to Azure Data Lake.
# Output is stored as Delta tables across Bronze, Silver, and Gold layers.

# COMMAND ----------

# Step 1: Import libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, explode, from_json, lit, to_date,
    year, month, dayofmonth
)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, TimestampType
from datetime import datetime, timedelta

# COMMAND ----------

# Step 2: Configure AWS S3 credentials via secret scope
spark._jsc.hadoopConfiguration().set("fs.s3a.access.key", dbutils.secrets.get(scope="aws-secrets", key="access-key"))
spark._jsc.hadoopConfiguration().set("fs.s3a.secret.key", dbutils.secrets.get(scope="aws-secrets", key="secret-key"))
spark._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "s3.amazonaws.com")

# COMMAND ----------


In [ ]:
# Step 3: Define S3 source and ADLS destination paths
s3_bucket = "s3a://your-s3-bucket-name"
source_path = f"{s3_bucket}/json_data/daily/{{date_partition}}.json"

adls_bronze = "abfss://bronze@your-storage-account.dfs.core.windows.net/data/json/"
adls_silver = "abfss://silver@your-storage-account.dfs.core.windows.net/data/processed/"
adls_gold = "abfss://gold@your-storage-account.dfs.core.windows.net/data/analytics/"

In [ ]:
# Step 4: Set process date as previous day
process_date = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')

# COMMAND ----------

# Step 5: Read raw nested JSON from S3
raw_df = spark.read.option("multiline", "true").json(source_path.format(date_partition=process_date))

In [ ]:
# Step 6: Display schema of nested JSON
raw_df.printSchema()

# COMMAND ----------

# Step 7: Flatten nested structure (example schema assumption)
# JSON fields assumed: user (struct), items (array of structs)
flattened_df = raw_df \
    .withColumn("user_id", col("user.id")) \
    .withColumn("user_name", col("user.name")) \
    .withColumn("created_at", to_date(col("created_at"))) \
    .withColumn("item", explode(col("items"))) \
    .withColumn("item_id", col("item.item_id")) \
    .withColumn("item_name", col("item.name")) \
    .withColumn("item_value", col("item.value")) \
    .drop("user", "items", "item")

In [ ]:
# COMMAND ----------

# Step 8: Add audit/metadata columns
flattened_df = flattened_df \
    .withColumn("ingestion_date", lit(process_date)) \
    .withColumn("source", lit("aws_s3")) \
    .withColumn("year", year("created_at")) \
    .withColumn("month", month("created_at")) \
    .withColumn("day", dayofmonth("created_at"))

# COMMAND ----------

# Step 9: Write raw data to Bronze Layer
raw_df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("created_at") \
    .save(adls_bronze + "transactions_raw")


In [ ]:
# COMMAND ----------

# Step 10: Write cleaned, flattened data to Silver Layer
flattened_df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("year", "month", "day") \
    .save(adls_silver + "transactions_cleaned")

# COMMAND ----------

# Step 11: Register Silver Delta table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS silver.transactions_cleaned
USING DELTA
LOCATION '{adls_silver}transactions_cleaned'
""")

# COMMAND ----------

In [ ]:
# Step 12: Sample Gold aggregation: total spend and item count per user
gold_df = flattened_df.groupBy("user_id", "user_name", "ingestion_date") \
    .agg({"item_value": "sum", "*": "count"}) \
    .withColumnRenamed("sum(item_value)", "total_spent") \
    .withColumnRenamed("count(1)", "item_count")

# COMMAND ----------

# Step 13: Write Gold Layer table
gold_df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("ingestion_date") \
    .save(adls_gold + "user_summary")

# COMMAND ----------

In [ ]:
# Step 14: Register Gold Delta table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS gold.user_summary
USING DELTA
LOCATION '{adls_gold}user_summary'
""")

# COMMAND ----------

# Step 15: Optimize Gold Layer for performance
spark.sql("OPTIMIZE gold.user_summary ZORDER BY (user_id)")

# COMMAND ----------

# Step 16: Preview Gold analytics result
display(spark.sql("SELECT * FROM gold.user_summary ORDER BY total_spent DESC LIMIT 20"))


In [ ]:
# COMMAND ----------

# Step 17: Optional - Retain only 7 days of files
spark.sql("VACUUM gold.user_summary RETAIN 168 HOURS")

# COMMAND ----------

# Step 18: Best practices notes (for documentation):
# - Use service principals for production environments
# - Secure secrets in Azure Key Vault
# - Parameterize process_date for job scheduling
# - Use Delta format throughout the pipeline
# - Handle schema evolution using 'mergeSchema' if needed

# COMMAND ----------

# Step 19: This notebook can be scheduled daily via Databricks Workflows
# Pass 'process_date' as a widget or notebook parameter for automation

# COMMAND ----------

# Step 20: End of ETL Pipeline
print("✅ ETL process complete for date:", process_date)